# 🏏 IPL 2026 — Live Patch + Dashboard
Drop this as **Section 2.5** in your integrated notebook, right after loading `matches.csv` and before feature engineering.

- Patches any matches played after your CSV download date
- Displays: Points Table, Recent Results, Upcoming Fixtures
- Auto-updates whenever you re-run the cell

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from IPython.display import display, HTML
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')     

SHORT_TO_FULL = {
    'MI':'Mumbai Indians','CSK':'Chennai Super Kings','RCB':'Royal Challengers Bangalore',
    'KKR':'Kolkata Knight Riders','DC':'Delhi Capitals','PBKS':'Punjab Kings',
    'RR':'Rajasthan Royals','SRH':'Sunrisers Hyderabad','GT':'Gujarat Titans','LSG':'Lucknow Super Giants',
}
FULL_TO_SHORT = {v:k for k,v in SHORT_TO_FULL.items()}
TEAM_COLORS   = {
    'MI':'#004BA0','CSK':'#F7A721','RCB':'#EC1C24','KKR':'#3A225D','DC':'#17449B',
    'PBKS':'#ED1B24','RR':'#E91E8C','SRH':'#FF6600','GT':'#1C4899','LSG':'#00A3E0',
}

# ── UPDATE PATH ──────────────────────────────────────────────────────────────
MATCHES_CSV = r'C:\Users\Vittu\Downloads\2026\matches.csv'
SAVE_DIR    = r'C:\Users\Vittu\Downloads'

print('✅ Setup ready')

## Step 1 — Load CSV + Patch Missing Matches
> **How to update:** When new matches are played, add a new row to `NEW_MATCHES` below and re-run this cell.

In [ ]:
# ── Load your downloaded CSV ──────────────────────────────────────────────────
m_csv = pd.read_csv(MATCHES_CSV)
m_csv['date'] = pd.to_datetime(m_csv['date'], format='%B %d, %Y', errors='coerce')
last_date = m_csv['date'].max()
print(f'📅 CSV last updated : {last_date.strftime("%B %d, %Y")} (Match {m_csv["match_id"].max()})')
print(f'📊 Matches in CSV   : {len(m_csv)}')

# ── MANUALLY ADD NEW MATCHES HERE ────────────────────────────────────────────
# Format: match_id, date (Month DD YYYY), venue, team1, team2,
#         toss_winner, toss_decision (Bat/Bowl),
#         first_ings_score, first_ings_wkts, second_ings_score, second_ings_wkts,
#         match_result, match_winner, wb_runs, wb_wickets, balls_left,
#         player_of_the_match, top_scorer, highscore, best_bowling, best_bowling_figure, super_over_match

NEW_MATCHES = [
    # ── Added: matches 40-42 played after Apr 27 download ────────────────────
    {
        'match_id':40,'date':'April 28, 2026',
        'venue':'Maharaja Yadavindra Singh Intl. Stadium, Mullanpur',
        'team1':'PBKS','team2':'RR','stage':'League',
        'toss_winner':'RR','toss_decision':'Bowl',
        'first_ings_score':222,'first_ings_wkts':4,
        'second_ings_score':228,'second_ings_wkts':4,
        'match_result':'completed','match_winner':'RR',
        'wb_runs':None,'wb_wickets':6,'balls_left':4,
        'player_of_the_match':'Vaibhav Sooryavanshi','top_scorer':'Prabhsimran Singh',
        'highscore':59,'best_bowling':'Jofra Archer','best_bowling_figure':'3-25','super_over_match':'No'
    },
    {
        'match_id':41,'date':'April 29, 2026',
        'venue':'Wankhede Stadium, Mumbai',
        'team1':'MI','team2':'SRH','stage':'League',
        'toss_winner':'MI','toss_decision':'Bat',
        'first_ings_score':243,'first_ings_wkts':5,
        'second_ings_score':249,'second_ings_wkts':4,
        'match_result':'completed','match_winner':'SRH',
        'wb_runs':None,'wb_wickets':6,'balls_left':8,
        'player_of_the_match':'Heinrich Klaasen','top_scorer':'Ryan Rickelton',
        'highscore':123,'best_bowling':'Eshan Malinga','best_bowling_figure':'2-54','super_over_match':'No'
    },
    {
        'match_id':42,'date':'April 30, 2026',
        'venue':'Narendra Modi Stadium, Ahmedabad',
        'team1':'GT','team2':'RCB','stage':'League',
        'toss_winner':'GT','toss_decision':'Bowl',
        'first_ings_score':155,'first_ings_wkts':10,
        'second_ings_score':158,'second_ings_wkts':6,
        'match_result':'completed','match_winner':'GT',
        'wb_runs':None,'wb_wickets':4,'balls_left':25,
        'player_of_the_match':'Jason Holder','top_scorer':'Devdutt Padikkal',
        'highscore':40,'best_bowling':'Bhuvneshwar Kumar','best_bowling_figure':'3-28','super_over_match':'No'
    },
    # ── ADD FUTURE MATCHES BELOW THIS LINE ───────────────────────────────────
    # Copy-paste the block above and update values after each match
]

# ── Merge new matches into CSV ────────────────────────────────────────────────
if NEW_MATCHES:
    df_new = pd.DataFrame(NEW_MATCHES)
    df_new['date'] = pd.to_datetime(df_new['date'], format='%B %d, %Y')
    # Only add matches not already in CSV
    existing_ids = set(m_csv['match_id'].tolist())
    df_new = df_new[~df_new['match_id'].isin(existing_ids)]
    matches = pd.concat([m_csv, df_new], ignore_index=True).sort_values('date').reset_index(drop=True)
    print(f'✅ Patched {len(df_new)} new match(es) → total {len(matches)} matches')
else:
    matches = m_csv.copy()
    print('ℹ️  No new matches to patch')

completed = matches[matches['match_result'] == 'completed'].copy()
print(f'✅ Completed matches : {len(completed)}')
print(f'📅 Date range        : {matches["date"].min().strftime("%b %d")} → {matches["date"].max().strftime("%b %d, %Y")}')

## Step 2 — Build Live Points Table

In [ ]:
def build_points_table(df):
    """Computes live points table from completed matches DataFrame."""
    rows = []
    teams = sorted(set(df['team1'].tolist() + df['team2'].tolist()))

    for t in teams:
        team_m = df[(df['team1']==t) | (df['team2']==t)]
        played = len(team_m)
        wins   = (team_m['match_winner'] == t).sum()
        abnd   = (team_m['match_result'] == 'abandoned').sum()
        losses = played - wins - abnd
        pts    = wins * 2 + abnd

        # NRR calculation
        runs_scored = runs_faced = balls_bowled = balls_faced = 0
        for _, r in team_m.iterrows():
            if r['match_result'] != 'completed': continue
            bat_first = r['team1'] if r.get('toss_decision','Bowl') == 'Bat' else r['team2']
            is_batting_first = (t == bat_first)
            if is_batting_first:
                runs_scored += float(r['first_ings_score'] or 0)
                balls_faced += 120
                runs_faced  += float(r['second_ings_score'] or 0)
                bl = float(r['balls_left'] or 0)
                balls_bowled += (120 - bl) if bl > 0 else 120
            else:
                runs_scored += float(r['second_ings_score'] or 0)
                bl = float(r['balls_left'] or 0)
                balls_faced += (120 - bl) if bl > 0 else 120
                runs_faced  += float(r['first_ings_score'] or 0)
                balls_bowled += 120

        nrr = 0.0
        if balls_faced > 0 and balls_bowled > 0:
            rr_scored = (runs_scored / balls_faced) * 6
            rr_faced  = (runs_faced  / balls_bowled) * 6
            nrr = rr_scored - rr_faced

        rows.append({
            'Team' : SHORT_TO_FULL.get(t, t),
            'Short': t,
            'M'    : played,
            'W'    : int(wins),
            'L'    : int(losses),
            'NR'   : int(abnd),
            'Pts'  : int(pts),
            'NRR'  : round(nrr, 3),
        })

    pt = pd.DataFrame(rows).sort_values(['Pts','NRR'], ascending=False).reset_index(drop=True)
    pt.index = pt.index + 1
    return pt


points_table = build_points_table(completed)

# ── Pretty print ──────────────────────────────────────────────────────────────
print('\n' + '='*72)
print(f'  🏆 IPL 2026 — POINTS TABLE  (Updated: {datetime.now().strftime("%d %b %Y %H:%M")})')
print('='*72)
print(f'  {"Pos":>3}  {"Team":<32} {"M":>3} {"W":>3} {"L":>3} {"NR":>3} {"Pts":>4} {"NRR":>7}')
print('  ' + '-'*65)
for pos, row in points_table.iterrows():
    qual  = '✅' if pos <= 4 else '  '
    nrr   = f"{row['NRR']:+.3f}"
    short = FULL_TO_SHORT.get(row['Team'], row['Short'])
    print(f'  {qual} {pos:>2}.  {short:<6}  {row["Team"]:<26} {row["M"]:>3} {row["W"]:>3} {row["L"]:>3} {row["NR"]:>3} {row["Pts"]:>4} {nrr:>7}')
print('  ' + '-'*65)
print('  ✅ = Playoff qualification zone (Top 4)')
print('='*72)

## Step 3 — Recent Results + Upcoming Matches

In [ ]:
# ── UPCOMING FIXTURES — update as schedule releases ──────────────────────────
UPCOMING = [
    {'no':43,'date':'May 01, 2026','teams':'RR vs DC',  't1':'RR', 't2':'DC',  'venue':'Sawai Mansingh Stadium, Jaipur',            'time':'7:30 PM IST'},
    {'no':44,'date':'May 02, 2026','teams':'CSK vs MI', 't1':'CSK','t2':'MI',  'venue':'M.A. Chidambaram Stadium, Chennai',         'time':'7:30 PM IST'},
    {'no':45,'date':'May 03, 2026','teams':'SRH vs KKR','t1':'SRH','t2':'KKR','venue':'Rajiv Gandhi Intl. Stadium, Hyderabad',      'time':'3:30 PM IST'},
    {'no':46,'date':'May 03, 2026','teams':'GT vs PBKS','t1':'GT', 't2':'PBKS','venue':'Narendra Modi Stadium, Ahmedabad',          'time':'7:30 PM IST'},
    {'no':47,'date':'May 04, 2026','teams':'RCB vs LSG','t1':'RCB','t2':'LSG','venue':'M. Chinnaswamy Stadium, Bengaluru',          'time':'7:30 PM IST'},
    {'no':48,'date':'May 05, 2026','teams':'KKR vs RR', 't1':'KKR','t2':'RR', 'venue':'Eden Gardens, Kolkata',                     'time':'7:30 PM IST'},
    {'no':49,'date':'May 06, 2026','teams':'MI vs DC',  't1':'MI', 't2':'DC',  'venue':'Wankhede Stadium, Mumbai',                  'time':'7:30 PM IST'},
    {'no':50,'date':'May 06, 2026','teams':'PBKS vs CSK','t1':'PBKS','t2':'CSK','venue':'Maharaja Y.S. Stadium, Mullanpur',         'time':'3:30 PM IST'},
]

today = pd.Timestamp.now().normalize()

print('\n' + '='*72)
print('  📋 RECENT RESULTS — Last 5 matches')
print('='*72)
recent5 = completed.sort_values('date').tail(5)
for _, r in recent5.iloc[::-1].iterrows():
    t1, t2 = r['team1'], r['team2']
    winner = r['match_winner']
    s1 = f"{int(r['first_ings_score'])}/{int(r['first_ings_wkts'])}"
    s2 = f"{int(r['second_ings_score'])}/{int(r['second_ings_wkts'])}"
    wkts = r.get('wb_wickets')
    runs = r.get('wb_runs')
    if pd.notna(wkts) and wkts > 0:
        margin = f'{int(wkts)} wickets'
    elif pd.notna(runs) and runs > 0:
        margin = f'{int(runs)} runs'
    else:
        margin = 'close'
    datestr = pd.to_datetime(r['date']).strftime('%d %b')
    print(f"  {datestr}  M{int(r['match_id']):<3}  {t1} vs {t2}")
    print(f"         {t1}: {s1}   {t2}: {s2}")
    print(f"         ✅ {winner} won by {margin}")
    print()

print('='*72)
print('  📅 UPCOMING FIXTURES')
print('='*72)
for u in UPCOMING:
    d = pd.to_datetime(u['date'])
    is_today = d.normalize() == today
    tag = ' ← TODAY' if is_today else ''
    print(f"  M{u['no']}  {d.strftime('%d %b')}  {u['teams']:<18}  {u['time']}  {u['venue']}{tag}")
print('='*72)

## Step 4 — Visual Dashboard (Matplotlib)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as ticker

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#0D1117')
gs  = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

DARK  = '#0D1117'
PANEL = '#161B22'
GRID  = '#21262D'
TEXT  = '#C9D1D9'
MUTED = '#8B949E'

def dark_ax(ax, title=''):
    ax.set_facecolor(PANEL)
    for spine in ax.spines.values():
        spine.set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    ax.yaxis.set_tick_params(color=GRID)
    ax.xaxis.set_tick_params(color=GRID)
    ax.grid(axis='y', color=GRID, linewidth=0.5, alpha=0.7)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, color=TEXT, fontsize=11, pad=10, fontweight='medium')

# ── Title banner ──────────────────────────────────────────────────────────────
ax_title = fig.add_subplot(gs[0, :])
ax_title.set_facecolor(DARK)
ax_title.axis('off')
ax_title.text(0.5, 0.7, '🏏  IPL 2026 — Live Dashboard',
              ha='center', va='center', fontsize=22, color=TEXT, fontweight='bold', transform=ax_title.transAxes)
ax_title.text(0.5, 0.2,
              f'Last updated: {datetime.now().strftime("%d %b %Y %H:%M")}   |   '
              f'Matches played: {len(completed)}   |   Remaining league matches: {70 - len(completed)}',
              ha='center', va='center', fontsize=11, color=MUTED, transform=ax_title.transAxes)

# ── 1. Points Table (left panel) ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1:, 0])
ax1.set_facecolor(PANEL)
for spine in ax1.spines.values(): spine.set_color(GRID)
ax1.axis('off')
ax1.set_title('Points Table', color=TEXT, fontsize=11, pad=10, fontweight='medium')

headers = ['Pos', 'Team', 'M', 'W', 'L', 'Pts', 'NRR']
col_x   = [0.04, 0.14, 0.55, 0.63, 0.71, 0.80, 0.91]

for cx, h in zip(col_x, headers):
    ax1.text(cx, 0.97, h, transform=ax1.transAxes, fontsize=8,
             color=MUTED, fontweight='medium', va='top')

row_h = 0.085
for i, (pos, row) in enumerate(points_table.iterrows()):
    y = 0.90 - i * row_h
    bg_color = '#1A2D1A' if pos <= 4 else PANEL
    ax1.add_patch(plt.Rectangle((0.01, y - 0.01), 0.98, row_h - 0.005,
                                 transform=ax1.transAxes, facecolor=bg_color,
                                 edgecolor='none', zorder=0))
    short = FULL_TO_SHORT.get(row['Team'], row['Team'][:4])
    tc    = TEAM_COLORS.get(short, '#888888')
    nrr_c = '#4CAF50' if row['NRR'] >= 0 else '#F44336'

    ax1.text(col_x[0], y + row_h/2 - 0.015, str(pos), transform=ax1.transAxes,
             fontsize=9, color=MUTED, va='center')
    ax1.add_patch(plt.Rectangle((col_x[1]-0.01, y + 0.01), 0.025, 0.04,
                                  transform=ax1.transAxes, facecolor=tc, edgecolor='none'))
    ax1.text(col_x[1]+0.025, y + row_h/2 - 0.015, short, transform=ax1.transAxes,
             fontsize=9, color=TEXT, va='center', fontweight='medium')
    ax1.text(col_x[2], y + row_h/2 - 0.015, str(row['M']),   transform=ax1.transAxes, fontsize=9, color=MUTED, va='center')
    ax1.text(col_x[3], y + row_h/2 - 0.015, str(row['W']),   transform=ax1.transAxes, fontsize=9, color=TEXT,  va='center', fontweight='medium')
    ax1.text(col_x[4], y + row_h/2 - 0.015, str(row['L']),   transform=ax1.transAxes, fontsize=9, color=MUTED, va='center')
    ax1.text(col_x[5], y + row_h/2 - 0.015, str(row['Pts']), transform=ax1.transAxes, fontsize=10, color='#F7A721', va='center', fontweight='bold')
    ax1.text(col_x[6], y + row_h/2 - 0.015, f"{row['NRR']:+.3f}",
             transform=ax1.transAxes, fontsize=9, color=nrr_c, va='center')

ax1.text(0.5, 0.01, '✅ Top 4 qualify for playoffs', transform=ax1.transAxes,
         fontsize=8, color='#4CAF50', ha='center', va='bottom')

# ── 2. Wins bar chart ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 1])
dark_ax(ax2, 'Wins this season')
pt_sorted = points_table.sort_values('W', ascending=False)
shorts = [FULL_TO_SHORT.get(t, t) for t in pt_sorted['Team']]
colors = [TEAM_COLORS.get(s, '#888') for s in shorts]
bars   = ax2.bar(shorts, pt_sorted['W'], color=colors, edgecolor='none', width=0.65)
for bar, val in zip(bars, pt_sorted['W']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(val), ha='center', fontsize=9, color=TEXT, fontweight='medium')
ax2.set_ylim(0, 9)
ax2.tick_params(axis='x', labelrotation=45, labelsize=8, colors=MUTED)
ax2.axhline(y=7, color='#F7A721', linestyle='--', linewidth=0.8, alpha=0.6)
ax2.text(9.5, 7.1, 'target ~7W', color='#F7A721', fontsize=7)

# ── 3. Recent results ─────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 2])
ax3.set_facecolor(PANEL)
for spine in ax3.spines.values(): spine.set_color(GRID)
ax3.axis('off')
ax3.set_title('Recent Results', color=TEXT, fontsize=11, pad=10, fontweight='medium')
recent = completed.sort_values('date').tail(5).iloc[::-1].reset_index(drop=True)
for i, r in recent.iterrows():
    y = 0.88 - i * 0.175
    t1, t2   = r['team1'], r['team2']
    winner   = r['match_winner']
    datestr  = pd.to_datetime(r['date']).strftime('%d %b')
    wkts     = r.get('wb_wickets')
    runs_wb  = r.get('wb_runs')
    margin   = f"{int(wkts)} wkts" if pd.notna(wkts) and wkts > 0 else (f"{int(runs_wb)} runs" if pd.notna(runs_wb) else '')
    s1 = f"{int(r['first_ings_score'])}/{int(r['first_ings_wkts'])}"
    s2 = f"{int(r['second_ings_score'])}/{int(r['second_ings_wkts'])}"
    tc1 = TEAM_COLORS.get(t1, '#888')
    tc2 = TEAM_COLORS.get(t2, '#888')
    ax3.add_patch(plt.Rectangle((0.02, y - 0.04), 0.96, 0.155,
                                  transform=ax3.transAxes, facecolor='#1C2430',
                                  edgecolor=GRID, linewidth=0.5, zorder=0))
    ax3.text(0.05, y + 0.08, f"M{int(r['match_id'])} — {datestr}",
             transform=ax3.transAxes, fontsize=7.5, color=MUTED, va='top')
    ax3.text(0.05, y + 0.04, t1, transform=ax3.transAxes,
             fontsize=9.5, color=tc1, va='top', fontweight='medium')
    ax3.text(0.28, y + 0.04, 'vs', transform=ax3.transAxes,
             fontsize=8, color=MUTED, va='top')
    ax3.text(0.35, y + 0.04, t2, transform=ax3.transAxes,
             fontsize=9.5, color=tc2, va='top', fontweight='medium')
    ax3.text(0.05, y - 0.01, f"{t1}: {s1}  |  {t2}: {s2}",
             transform=ax3.transAxes, fontsize=8, color=MUTED, va='top')
    win_c = TEAM_COLORS.get(winner, '#4CAF50')
    ax3.text(0.05, y - 0.025, f"✅ {winner} won {margin}",
             transform=ax3.transAxes, fontsize=8, color='#4CAF50', va='top', fontweight='medium')

# ── 4. Upcoming fixtures ──────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 1:])
ax4.set_facecolor(PANEL)
for spine in ax4.spines.values(): spine.set_color(GRID)
ax4.axis('off')
ax4.set_title('Upcoming Fixtures', color=TEXT, fontsize=11, pad=10, fontweight='medium')

uf = UPCOMING[:8]
cols = 2
rows_up = (len(uf) + cols - 1) // cols
for i, u in enumerate(uf):
    col   = i % cols
    row_i = i // cols
    x     = 0.03 + col * 0.5
    y     = 0.88 - row_i * 0.22
    d     = pd.to_datetime(u['date'])
    is_today_flag = d.normalize() == today
    bg_c  = '#1A2040' if is_today_flag else '#1C2430'
    lc    = '#EC1C24' if is_today_flag else GRID
    ax4.add_patch(plt.Rectangle((x, y - 0.07), 0.46, 0.18,
                                  transform=ax4.transAxes, facecolor=bg_c,
                                  edgecolor=lc, linewidth=0.8 if is_today_flag else 0.5))
    label = 'TODAY' if is_today_flag else d.strftime('%d %b')
    ax4.text(x + 0.02, y + 0.07, f"M{u['no']} — {label}",
             transform=ax4.transAxes, fontsize=8, color='#EC1C24' if is_today_flag else MUTED, va='top')
    t1c = TEAM_COLORS.get(u['t1'], '#888')
    t2c = TEAM_COLORS.get(u['t2'], '#888')
    ax4.text(x + 0.02, y + 0.025, u['t1'], transform=ax4.transAxes,
             fontsize=10, color=t1c, va='top', fontweight='medium')
    ax4.text(x + 0.10, y + 0.025, 'vs', transform=ax4.transAxes,
             fontsize=8, color=MUTED, va='top')
    ax4.text(x + 0.16, y + 0.025, u['t2'], transform=ax4.transAxes,
             fontsize=10, color=t2c, va='top', fontweight='medium')
    ax4.text(x + 0.02, y - 0.03, f"{u['time']}  |  {u['venue'][:35]}",
             transform=ax4.transAxes, fontsize=7.5, color=MUTED, va='top')

plt.savefig(f'{SAVE_DIR}\\ipl_2026_dashboard.png', dpi=130, bbox_inches='tight',
            facecolor=DARK, edgecolor='none')
plt.show()
print(f'💾 Dashboard saved → {SAVE_DIR}\\ipl_2026_dashboard.png')

## Step 5 — How to Keep Data Fresh

```
AFTER EACH MATCH DAY:
  1. Open this notebook
  2. Add the new match dict to NEW_MATCHES list in Step 1
  3. Add the result date/teams/scores/winner/margin
  4. Re-run all cells — points table, dashboard, predictions all auto-update

TO ADD AN UPCOMING FIXTURE:
  Update the UPCOMING list in Step 3 with the next match details

FIELDS TO FILL for each new match in NEW_MATCHES:
  match_id        → next number (e.g. 43)
  date            → 'May 01, 2026'
  team1/team2     → short names: MI, CSK, RCB etc.
  match_winner    → short name of winner
  wb_wickets      → wickets margin (or None if won by runs)
  wb_runs         → runs margin (or None if won by wickets)
  first/second_ings_score/wkts → scores from scorecard
```